# Missing Data

**Topic:** Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, HBox, VBox, HTML

from IPython.display import display, clear_output
from scipy import stats

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Identify** which columns in a dataset have missing values and how much is missing
- **Explain** the difference between data that is missing at random versus missing for a reason
- **Evaluate** the tradeoffs between the most common strategies for handling missing values

> **Tip:** Select the `cabin` column and try each strategy in turn. With 77% missing, none of the strategies is obviously right. That tension is exactly what makes missing data interesting.

---
## How we got here

In `03_data_types_and_casting.ipynb` we examined what types each column should have. Before we can clean and prepare the data for modeling, we need to confront what is absent. Missing data is one of the most common and most consequential data quality problems in real-world datasets.

---
## Why this matters for data science

Most machine learning algorithms cannot handle missing values. If you pass a DataFrame with NaN values to scikit-learn, you will get an error. But the solution is not simply to delete rows or fill with zeros. The right strategy depends on why the data is missing, how much is missing, and whether the missingness itself carries information. Getting this wrong introduces bias that is invisible in training metrics but visible in production failures.

---
## Try it yourself

In [ ]:
out = Output()

MISSING_COLS = [c for c in titanic.columns if titanic[c].isnull().any()]

STRATEGIES = {
    "Drop rows with missing values": {
        "description": "Remove every row that has a missing value in this column. Simple and clean, but you lose data. Best when missing values are rare.",
        "fn": lambda df, col: df.dropna(subset=[col])[[col]].describe() if df[col].dtype in ["float64", "int64"] else df.dropna(subset=[col])[[col]].value_counts().head(5),
    },
    "Fill with mean (numeric only)": {
        "description": "Replace missing values with the column mean. Keeps all rows. Works for numeric columns with roughly symmetric distributions.",
        "fn": lambda df, col: df[[col]].fillna(df[col].mean()) if df[col].dtype in ["float64", "int64"] else None,
    },
    "Fill with mode (most common value)": {
        "description": "Replace missing values with the most frequent value. Works for both numeric and categorical columns.",
        "fn": lambda df, col: df[[col]].fillna(df[col].mode()[0]).value_counts().head(5),
    },
    "Leave as missing (NaN)": {
        "description": "Do nothing for now. Some models (like XGBoost) can handle missing values natively. Useful when missingness itself may carry information.",
        "fn": lambda df, col: df[[col]].isnull().value_counts(),
    },
}

col_dropdown = Dropdown(
    options=MISSING_COLS,
    value="age",
    description="Column:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"),
)

strategy_dropdown = Dropdown(
    options=list(STRATEGIES.keys()),
    value="Drop rows with missing values",
    description="Strategy:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="400px"),
)

def render(col, strategy_name):
    n_missing = titanic[col].isnull().sum()
    pct_missing = n_missing / len(titanic) * 100
    strategy = STRATEGIES[strategy_name]
    result = strategy["fn"](titanic.copy(), col)

    summary_html = HTML(
        f'<div style="font-family: Inter, Arial, sans-serif; font-size: 14px; margin-bottom: 10px;">'
        f'<strong>{col}</strong>: {n_missing} missing values ({pct_missing:.1f}% of 891 rows)<br>'
        f'<span style="color: #868E96;">{strategy["description"]}</span>'
        f'</div>'
    )
    with out:
        clear_output(wait=True)
        display(summary_html)
        if result is not None:
            display(result)
        else:
            display(HTML('<em style="color: #868E96;">Mean fill only applies to numeric columns.</em>'))

def on_change(change):
    render(col_dropdown.value, strategy_dropdown.value)

col_dropdown.observe(on_change, names="value")
strategy_dropdown.observe(on_change, names="value")

display(VBox([HBox([col_dropdown, strategy_dropdown]), out]))
render(col_dropdown.value, strategy_dropdown.value)

---
## What's happening?

Missing data in pandas appears as `NaN` (Not a Number). The key questions to ask about any missing column are:

| Question | Why it matters |
|----------|---------------|
| How much is missing? | A column that is 5% missing is handled differently than one that is 77% missing |
| Is the missingness random? | Random missingness or Missing Completely at Random (MCAR) allows mean/mode fill. Systematic missingness or Missing Not at Random (MNAR) does not |
| Does absence mean something? | For `cabin`, having no cabin recorded might mean the passenger was in 3rd class |
| What happens to the rows? | Dropping rows costs you data. Filling creates assumptions |

The three main strategies:

```python
# Drop rows with missing values in a column
df.dropna(subset=['age'])

# Fill with the mean (numeric columns)
df['age'].fillna(df['age'].mean())

# Fill with the mode (most frequent value — works for categories too)
df['embarked'].fillna(df['embarked'].mode()[0])
```

---
## Real-world example: Titanic Dataset

The chart below shows the percentage of missing values in each affected column. Most columns are complete. But `age`, `cabin`, and `embarked` each require a deliberate decision.

Notice:
- **`cabin`** at 77% missing is the most extreme case. More data is absent than present
- **`age`** at about 20% missing is substantial but manageable with imputation
- **`embarked`** is 99.8% complete, with just 2 missing rows that can safely be dropped

> **Discussion question:** The `cabin` column is 77% missing. Should we drop it entirely or keep it? What information might the presence or absence of a cabin number tell us about the passenger?

In [ ]:
missing_pct = (titanic.isnull().sum() / len(titanic) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig = go.Figure(
    data=[go.Bar(
        x=missing_pct.index.tolist(),
        y=missing_pct.values,
        marker_color=[
            PALETTE["secondary"] if v > 50 else PALETTE["primary"] if v > 10 else PALETTE["muted"]
            for v in missing_pct.values
        ],
        text=[f"{v:.1f}%" for v in missing_pct.values],
        textposition="outside",
    )],
    layout=base_layout(
        title="Percentage of Missing Values per Column (Titanic)",
        xaxis_title="Column",
        yaxis_title="% Missing",
    ),
)
fig.update_layout(
    yaxis=dict(range=[0, 90]),
    showlegend=False,
)
fig.show()

### Missing data strategies across real-world scenarios

| Scenario | Strategy | Reason |
|----------|----------|--------|
| Patient age in medical records (5% missing) | Fill with median | Small amount, median avoids outlier sensitivity |
| Customer churn flag (0% should be missing) | Investigate source | Missing outcomes suggest a data pipeline problem |
| Product review text (30% missing) | Create binary "has_review" feature | Absence is informative and should be preserved |
| Sensor readings with gaps | Interpolate (time-based) | Temporal context makes interpolation meaningful |
| Survey answer to optional question | Leave as NaN | Unanswered questions are a valid response category |

---
## Key takeaway

> **How you handle missing data is one of the highest-stakes decisions in the entire EDA process, because every strategy introduces an assumption about why the data is absent.**

---
*Next up: 05_handling_duplicates.ipynb — checking for duplicate records and understanding what to do when you find them*